In [47]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

# Loader

In [37]:
loader = TextLoader("../Services/VectorDatabase/documents/disqualified.txt")
docs = loader.load()

In [38]:
def get_sentence_level_text_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=1,  # ensures each sentence is treated as its own chunk
        chunk_overlap=0,  # No overlap
        separators=[". ", "! ", "? "]  # Sentence-level separators
    )

text_splitter = get_sentence_level_text_splitter()

In [39]:
splitted_docs = text_splitter.split_documents(docs)

In [40]:
len(splitted_docs)

40

In [41]:
splitted_docs[7].page_content

'. A smiling servant filled the glasses of Tardo and Peo.\n"You see, there was no fuel for the ship to explore other planets in the\nsystem, and the ship just rusted away'

In [55]:
type(splitted_docs)

list

# Embedding model

In [43]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "thenlper/gte-small",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},  # for cosine similarity
)

/home/s448780/miniconda3/envs/synk/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [44]:
embedding_model

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-small', cache_folder=None, model_kwargs={'device': 'cuda'}, encode_kwargs={'normalize_embeddings': True}, multi_process=False, show_progress=False)

# Faiss DB

In [58]:
v_db = FAISS.from_documents(
            splitted_docs, 
            embedding_model, 
            distance_strategy=DistanceStrategy.COSINE
        )

In [64]:
query = "Who is Tardo?"

In [65]:
e_query = embedding_model.embed_query(query)

In [67]:
v_db.similarity_search_with_score(query, top_k = 10)

[(Document(metadata={'source': '../Services/VectorDatabase/documents/disqualified.txt'}, page_content='. They have a union, you know."\n\nTardo laughed.\n\n"A carry-over from Earth, no doubt," he commented'),
  0.23660251),
 (Document(metadata={'source': '../Services/VectorDatabase/documents/disqualified.txt'}, page_content='. "I\'ve seen enough."\n\n"Why?" asked Peo, surprised.\n\n"There are two classes of people on this planet, and we\'ve seen only\none," said Tardo'),
  0.24712801),
 (Document(metadata={'source': '../Services/VectorDatabase/documents/disqualified.txt'}, page_content=". FONTENAY\n\n\nAfter the morning inspection tour, Tardo, the Solar Council's Planetary\nAid agent, and his companion, Peo, were taken to the castle which stood\non a hill overlooking the area.\n\nTardo and Peo were entertained royally at luncheon by Saranta, their\nhost, who appeared to be the wealthy overlord of this portion of the\nplanet"),
  0.25594065),
 (Document(metadata={'source': '../Services/